<a href="https://colab.research.google.com/github/carlosmonge266-hub/sesion-1-ia/blob/main/Ejercicio4final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import sys

# Force reinstall all key packages to ensure fresh versions and resolve potential conflicts
!{sys.executable} -m pip install -q --force-reinstall gradio langchain langchain-google-genai unstructured chromadb sentence-transformers pypdf langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata

# Set the GOOGLE_API_KEY as an environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
import google.generativeai as genai
import os

# Ensure your GOOGLE_API_KEY is set in os.environ
# (It should be set by the previous cell)

try:
    print("Listing available embedding models:")
    for m in genai.list_models():
        if "embedContent" in m.supported_generation_methods:
            print(m.name)
except Exception as e:
    print(f"Error listing models: {e}")

Listing available embedding models:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Updated import

# Provide the path to your PDF document(s)
# For example, if you uploaded 'my_document.pdf' to the /content directory:
# To upload your PDF:
# 1. Click the 'Files' icon (folder icon) on the left sidebar.
# 2. Click the 'Upload to session storage' icon (document with an arrow pointing up).
# 3. Select your PDF file.
# 4. Once uploaded, you can reference it as `/content/your_document_name.pdf`.
pdf_path = "/content/sample_data/Asignación de Objetivos - Rol Manager 1.pdf" # <--- CHANGE THIS TO YOUR UPLOADED DOCUMENT PATH

# Load the document
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Split the documents into smaller chunks for processing
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} documents and split into {len(splits)} chunks.")
# print(f"First chunk: {splits[0].page_content}")

Loaded 12 documents and split into 13 chunks.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize Google Generative AI Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001") # Changed model name to an available one

# Create a Chroma vector store from the document splits
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print("Vector store created successfully with document embeddings.")

Vector store created successfully with document embeddings.


## 3. Set up RAG Chain

Now we'll integrate Google's Gemini model with our vector store using LangChain to create a Retrieval-Augmented Generation (RAG) chain. This chain will retrieve relevant document chunks based on a query and then use Gemini to generate a response informed by those chunks.

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.3)

# Create a retriever from our vector store
retriever = vectorstore.as_retriever()

# Define the prompt for our RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the provided context, making sure to cite your sources if possible. If you don't know the answer, state that you don't know."),
    ("user", "Context: {context}\nQuestion: {input}")
])

# Create a chain to combine documents
document_chain = create_stuff_documents_chain(llm, prompt)

# Create the retrieval chain
rag_chain = create_retrieval_chain(retriever, document_chain)

print("RAG chain successfully set up!")

ModuleNotFoundError: No module named 'langchain.chains'

## 4. Build Gradio ChatInterface

Finally, we'll create the Gradio application using `gr.ChatInterface`. This will provide an interactive chat experience where you can ask questions, and the RAG system will respond, indicating the sources it used.

In [ ]:
import gradio as gr

# Define a function to handle chat messages
async def chat_with_rag(message, history):
    response = rag_chain.invoke({"input": message})
    answer = response["answer"]
    sources = set()
    for doc in response["context"]:
        if 'source' in doc.metadata:
            sources.add(doc.metadata['source'])

    if sources:
        answer += "\n\n**Sources:**\n" + "\n".join(f"- {s}" for s in sources)

    return answer

# Create the Gradio ChatInterface
iface = gr.ChatInterface(
    chat_with_rag,
    chatbot=gr.Chatbot(height=500, allow_tags=False),
    textbox=gr.Textbox(placeholder="Ask me a question about your documents...", container=False, scale=7),
    title="RAG System with LangChain, Gemini, and Gradio",
    description="Ask questions about the documents you've provided. The system will retrieve relevant information and generate answers, citing its sources.",
    # Removed 'theme' argument as it's no longer supported in this Gradio version
    examples=["What is this document about?"],
    cache_examples=False
)

# Launch the Gradio app
iface.launch(share=True)

Exception in thread Thread-9 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "uvloop/loop.pyx", line 1518, in uvloop.loop.Loop.run_until_complete
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 79, in serve
    await s